# Introduction to GPUs: Colab Hands-On Lab

This notebook compares NumPy on CPU, PyTorch on CPU, and PyTorch on GPU. Run it in Google Colab with **Runtime > Change runtime type > GPU**.

The goal is not to produce one universal benchmark number. The goal is to understand when a GPU helps, what data movement costs, and why we must measure real workloads.

## Questions before we start

1. What do you expect to be faster: adding a few numbers on the CPU or moving them to a GPU?
2. What changes when we multiply large matrices?
3. Should transfer time be included in a production benchmark?

Write down your predictions before running the cells.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device('cuda')
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    device = torch.device('cpu')
    print('No GPU detected. The notebook will still run using the CPU.')

## Experiment 1: Vector addition

Each output value is independent: `c[i] = a[i] + b[i]`. This is a simple example of data parallelism. The GPU may not win here because the operation does little arithmetic compared with the cost of moving data and launching work.

In [ ]:
def seconds(function):
    start = time.perf_counter()
    result = function()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.perf_counter() - start, result

n = 10_000_000
np_a = np.random.rand(n).astype(np.float32)
np_b = np.random.rand(n).astype(np.float32)
torch_a_cpu = torch.from_numpy(np_a)
torch_b_cpu = torch.from_numpy(np_b)
torch_a_gpu = torch_a_cpu.to(device)
torch_b_gpu = torch_b_cpu.to(device)

np_time, np_result = seconds(lambda: np_a + np_b)
torch_cpu_time, torch_cpu_result = seconds(lambda: torch_a_cpu + torch_b_cpu)
torch_gpu_time, torch_gpu_result = seconds(lambda: torch_a_gpu + torch_b_gpu)

vector_results = pd.DataFrame({
    'implementation': ['NumPy CPU', 'PyTorch CPU', 'PyTorch GPU' if device.type == 'cuda' else 'PyTorch fallback CPU'],
    'seconds': [np_time, torch_cpu_time, torch_gpu_time],
})
display(vector_results)
print('Results agree:', np.allclose(np_result, torch_cpu_result.numpy()))

### Discuss

- Did the result match your prediction?
- Was the GPU time measuring computation only, or did it include the initial data transfer?
- Why might vector addition be limited by memory movement rather than arithmetic?

The tensors were moved to the GPU before the timed operation. This demonstrates kernel execution time, not complete application time. We measure transfer explicitly later.

## Experiment 2: Matrix multiplication

Matrix multiplication performs much more arithmetic and gives the GPU more parallel work. Increase the size carefully if your GPU has limited memory.

In [ ]:
def benchmark(function, repeats=5, warmups=2):
    for _ in range(warmups):
        function()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    measurements = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = function()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        measurements.append(time.perf_counter() - start)
    return float(np.median(measurements)), result

size = 2048
np_left = np.random.rand(size, size).astype(np.float32)
np_right = np.random.rand(size, size).astype(np.float32)
torch_left_cpu = torch.from_numpy(np_left)
torch_right_cpu = torch.from_numpy(np_right)
torch_left_gpu = torch_left_cpu.to(device)
torch_right_gpu = torch_right_cpu.to(device)

np_matmul_time, _ = benchmark(lambda: np.matmul(np_left, np_right))
torch_cpu_matmul_time, _ = benchmark(lambda: torch.matmul(torch_left_cpu, torch_right_cpu))
torch_gpu_matmul_time, _ = benchmark(lambda: torch.matmul(torch_left_gpu, torch_right_gpu))

matmul_results = pd.DataFrame({
    'implementation': ['NumPy CPU', 'PyTorch CPU', 'PyTorch GPU' if device.type == 'cuda' else 'PyTorch fallback CPU'],
    'median_seconds': [np_matmul_time, torch_cpu_matmul_time, torch_gpu_matmul_time],
})
display(matmul_results)
display(matmul_results.assign(relative_to_numpy=matmul_results.median_seconds / np_matmul_time))

In [ ]:
matmul_results.plot.bar(x='implementation', y='median_seconds', legend=False, title='Matrix multiplication timing')
plt.ylabel('Seconds, lower is better')
plt.xticks(rotation=0)
plt.show()

### Discuss

- At this size, did the GPU have enough work to show its strength?
- Why do we use warm-up runs?
- Why do we synchronize before reading a GPU timing?
- Would this speedup alone justify moving an entire application to the cloud?

## Experiment 3: Data-transfer overhead

This measures the time required to copy two matrices from CPU memory to GPU memory. Real applications should include this cost when deciding whether acceleration is worthwhile.

In [ ]:
if torch.cuda.is_available():
    transfer_time, _ = benchmark(lambda: (torch_left_cpu.to('cuda'), torch_right_cpu.to('cuda')), repeats=5)
    print(f'Median transfer time: {transfer_time:.6f} seconds')
    print(f'Median GPU matmul time: {torch_gpu_matmul_time:.6f} seconds')
    print(f'One transfer plus one matmul: {transfer_time + torch_gpu_matmul_time:.6f} seconds')
else:
    print('GPU unavailable, so there is no CPU-to-GPU transfer to measure.')

## Change the experiment

Change `size` to 256, 512, 1024, and 2048. Record the results. Do not use a size that exhausts your GPU memory.

| Matrix size | NumPy CPU | PyTorch CPU | PyTorch GPU | Transfer time |
| --- | ---: | ---: | ---: | ---: |
| 256 | | | | |
| 512 | | | | |
| 1024 | | | | |
| 2048 | | | | |

### Reflection questions

1. At what size did the GPU become useful?
2. Did the same winner appear for every operation?
3. What would happen if the application transferred each row separately?
4. Which parts of this notebook would need to change for a production job?

## From Colab to production

In a paid production-focused course, this PyTorch computation can become a batch service:

```text
User or API -> AWS Lambda -> AWS Batch GPU job -> S3 results
```

Lambda submits and coordinates the job. The GPU computation runs in an AWS Batch job on GPU-enabled compute resources. Lambda itself is not a GPU runtime.

Further reading:
- [AWS Batch GPU jobs](https://docs.aws.amazon.com/batch/latest/userguide/gpu-jobs.html)
- [PyTorch CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html)
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)